# AI Insurance Claim Assistant

## Project

Build an AI Insurance Claim Assistant capable of:

- Understanding patient requests
- Looking up policy information
- Estimating claim amount
- Detecting missing documents
- Planning multiple actions
- Calling multiple tools
- Performing multi-step reasoning
- Making dynamic decisions
- Understanding text, images and audio

---

## Concepts Covered

### Building Multi-Tool Agent Workflows

- Multi-Step Reasoning
- Tool Chaining
- Workflow Execution
- Planning Multiple Actions
- Dynamic Decision Making
- Multi-Tool Workflow

---

### Building Multi-Modal Applications

- Text
- Images
- Audio
- Combining Multiple Modalities

In [1]:
from openai import OpenAI
import base64
import os

client = OpenAI(api_key = os.getenv("OPENAI_API_KEY"))

# Building Multi-Tool Agent Workflow

The assistant has access to multiple tools.

Instead of answering immediately, it decides:

1. Which tool to use
2. In what order
3. Whether another tool is needed
4. When to stop

This demonstrates

- Tool Chaining
- Workflow Execution
- Multi-Step Reasoning
- Dynamic Decision Making

In [2]:
# too1
def policy_lookup(policy_id):
    policies = {
        "P100":{
            "name":"John",
            "coverage":"Premium",
            "limit":1000000
        },
        "P200":{
            "name":"Alice",
            "coverage":"Basic",
            "limit": 300000
        }
    }

    return policies.get(policy_id, "ERROR: POLICY ID NOT FOUND.")

In [3]:
## tool2
def estimate_claim(hospital_bill):
    return hospital_bill * 0.89

In [4]:
## tool3 - missing docs checker
def check_documents(documents):
    required = {
        "ID Proof",
        "Hospital Bill",
        "Discharge Summary"
    }

    missing = []
    for doc in required:
        if doc not in documents:
            missing.append(doc)
    
    return missing

# Multi-Step Reasoning

Instead of directly approving a claim, the assistant reasons through multiple steps.

Example

Step 1 → Read user request

↓

Step 2 → Lookup policy

↓

Step 3 → Estimate claim

↓

Step 4 → Check documents

↓

Step 5 → Make final decision

In [6]:
## LLM planner
def planner(user_query):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages = [
            {
                "role":"system",
                "content":""" 
You are an Insurance Claim Planner.

Return only the tools to execute.

possbile tools:
1. policy_lookup
2. estimate_claim
3. check_documents
"""
            },
            {
                "role":"user",
                "content": user_query
            }
        ]
    )
    return response.choices[0].message.content

# Workflow Execution

The workflow executes multiple tools one after another.

This is called Tool Chaining.

In [7]:
## workflow
policy = policy_lookup("P100")
claim = estimate_claim(250000)
missing = check_documents(["ID Proff",
                           "Hospital Bill"])

print(policy)
print(claim)
print(missing)

{'name': 'John', 'coverage': 'Premium', 'limit': 1000000}
222500.0
['ID Proof', 'Discharge Summary']


# Dynamic Decision Making

The workflow changes based on the intermediate outputs.

Example

If policy not found

↓

Stop workflow

Else

↓

Estimate claim

↓

Check documents

↓

Approve or Reject

In [8]:
policy = policy_lookup("P100")

if policy == "Policy Not Found":
    print("Claim Rejected")
else:
    claim = estimate_claim(250000)

    missing = check_documents(['ID Proof', 'Hospital Bill'])

    if missing:
        print("Missing Documents")
        print(missing)
    else:
        print("claim Approved")
        print("Estimated Amount:", claim)

Missing Documents
['Discharge Summary']


# Multi-Tool Insurance Assistant Workflow

The assistant plans multiple actions.

User Request

↓

Policy Lookup Tool

↓

Claim Estimation Tool

↓

Document Verification Tool

↓

Final Response

In [11]:
def insurance_assistant(policy_id, bill_amount, documents):

    print("Step 1 : Policy Lookup")

    policy = policy_lookup(policy_id)

    print(policy)

    if policy == "Policy Not Found":

        print("\nWorkflow Stopped")

        return

    print("\nStep 2 : Estimate Claim")

    claim = estimate_claim(bill_amount)

    print(claim)

    print("\nStep 3 : Document Verification")

    missing = check_documents(documents)

    if missing:

        print("Missing Documents")

        print(missing)

        return

    print("\nFinal Decision")

    print("Claim Approved")

    print("Approved Amount :",claim)

In [12]:
insurance_assistant(

    "P100",

    250000,

    [
        "ID Proof",
        "Hospital Bill",
        "Discharge Summary"
    ]

)

Step 1 : Policy Lookup
{'name': 'John', 'coverage': 'Premium', 'limit': 1000000}

Step 2 : Estimate Claim
222500.0

Step 3 : Document Verification

Final Decision
Claim Approved
Approved Amount : 222500.0


# Building Multi-Modal Applications

Multi-Modality means an AI system can understand more than one type of input.

Examples

- Text
- Images
- Audio

A single workflow can combine all of them before making a decision.

In [13]:
def analyze_medical_bill(image_path):

    with open(image_path, "rb") as f:
        image = base64.b64encode(f.read()).decode()

    response = client.chat.completions.create(

        model="gpt-4.1",

        messages=[

            {
                "role":"user",
                "content":[

                    {
                        "type":"text",
                        "text":"Extract hospital bill details."
                    },

                    {
                        "type":"image_url",
                        "image_url":{
                            "url":f"data:image/png;base64,{image}"
                        }
                    }

                ]
            }

        ]

    )

    return response.choices[0].message.content

In [14]:
def transcribe_audio(audio_path):

    with open(audio_path,"rb") as audio:

        transcript = client.audio.transcriptions.create(

            model="gpt-4o-mini-transcribe",

            file=audio

        )

    return transcript.text

In [15]:
def multimodal_claim(text, image_path, audio_path):

    print("Patient Text")

    print(text)

    print("\nMedical Bill")

    bill = analyze_medical_bill(image_path)

    print(bill)

    print("\nPatient Voice")

    voice = transcribe_audio(audio_path)

    print(voice)

    print("\nFinal Summary")

    response = client.chat.completions.create(

        model="gpt-4.1-mini",

        messages=[

            {
                "role":"system",
                "content":"Summarize the insurance claim."
            },

            {
                "role":"user",
                "content":f"""
Patient Text:
{text}

Bill:
{bill}

Voice:
{voice}
"""
            }

        ]

    )

    print(response.choices[0].message.content)